<h1 style="text-align: center;">TÉCNICO EM CIÊNCIA DE DADOS</h1>
<h1 style="text-align: center;">Roteiro do Desenvolvendo Juntos</h1>
<br>
<br>

**Componente:** Aprendizagem de Máquina
<br>
**Unidade Curricular:** Modelos, Algoritmos e Estimadores
<br>
**Tema da Semana:** Modelos, Algoritmos e Estimadores
<br>
**Semana 6**
<br>
**Aula 3:**  Aplicações de melhoria de modelos


<h2 align="center"> Mão na Massa: Aplicando Regularização
</h2>

Execute o código abaixo para instalar as bibliotecas (se for necessário) para a aula.

In [9]:
%pip install pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Prática Guiada Passo a Passo

Cada etapa deve ser demonstrada pelo professor antes de os alunos repetirem, garantindo compreensão e conexão com a rotina de um profissional da área. Explicação detalhada de cada comando, função ou ação, para que os alunos compreendam o propósito e a aplicação prática.

### Etapa 1: Preparação do Ambiente e Dados
Vamos carregar o dataset, criar o ruído e padronizar os dados (escalonamento). A padronização é essencial porque o Lasso penaliza os coeficientes com base em sua magnitude.

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler # Novo import necessário

# Carregando o dataset real
dados = pd.read_csv('USA_housing_dataset_pt_br.csv')

# Selecionando as variáveis principais e criando um ruído aleatório
X = dados[['quartos', 'banheiros', 'area_util_pes2', 'ano_construcao']].copy()
X['variavel_ruido'] = np.random.randint(1, 100, size=len(X)) # Variável inútil

y = dados['preco']

# Dividindo os dados (80% treino, 20% teste)
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

# Padronizando os dados 
scaler = StandardScaler()
X_treino_scaled = scaler.fit_transform(X_treino)
X_teste_scaled = scaler.transform(X_teste)

print("Dataset carregado, dividido e padronizado!")

Dataset carregado, dividido e padronizado!


### Etapa 2: O Problema do Modelo Tradicional (Sem Regularização)

Nesta etapa, treinamos a Regressão Linear comum. O modelo tentará atribuir peso a todas as colunas, inclusive ao ruído, o que pode prejudicar sua capacidade de generalização.

In [11]:
modelo_comum = LinearRegression()
modelo_comum.fit(X_treino_scaled, y_treino)

erro_treino = mean_absolute_error(y_treino, modelo_comum.predict(X_treino_scaled))
erro_teste = mean_absolute_error(y_teste, modelo_comum.predict(X_teste_scaled))

print(f"Erro Treino (Sem Ajuste): US$ {erro_treino:.2f}")
print(f"Erro Teste (Sem Ajuste): US$ {erro_teste:.2f}")

Erro Treino (Sem Ajuste): US$ 8030.47
Erro Teste (Sem Ajuste): US$ 7581.70


### Etapa 3: Melhorando com Lasso
Agora aplicamos a regressão Lasso. A regularização vai penalizar variáveis que não ajudam na previsão, reduzindo o peso do "ruído" a zero (ou próximo disso) e focando no que realmente define o preço do imóvel.

In [12]:
# Com os dados padronizados, o alpha age de forma equilibrada
modelo_lasso = Lasso(alpha=5000.0) 
modelo_lasso.fit(X_treino_scaled, y_treino)

erro_treino_lasso = mean_absolute_error(y_treino, modelo_lasso.predict(X_treino_scaled))
erro_teste_lasso = mean_absolute_error(y_teste, modelo_lasso.predict(X_teste_scaled))

print(f"Erro Treino (Lasso): US$ {erro_treino_lasso:.2f}")
print(f"Erro Teste (Lasso): US$ {erro_teste_lasso:.2f}")

# Mostrando os pesos
pesos = pd.DataFrame({'Variável': X.columns, 'Peso (Coeficiente)': modelo_lasso.coef_})
print("\nPesos do Modelo Lasso:\n", pesos)

Erro Treino (Lasso): US$ 9635.99
Erro Teste (Lasso): US$ 9053.44

Pesos do Modelo Lasso:
          Variável  Peso (Coeficiente)
0         quartos        14353.734329
1       banheiros        24450.620263
2  area_util_pes2       191300.903118
3  ano_construcao        25641.579015
4  variavel_ruido           -0.000000
